In [63]:
class DoubleBuffer:
    def __init__(self):
        self.left_size = 0
        self.right_size = 0
        self.left_items = []
        self.right_items = []
        
    def __repr__(self):
        return f'Left: {self.left_items} | Right: {self.right_items}'
        
    def num_letters(self):
        return self.left_size + self.right_size
    
    def num_items(self):
        return len(self.left_items) + len(self.right_items)
    
    def add_left(self, word):
        self.left_size += len(word)
        self.left_items.append(word)
        
    def add_right(self, word):
        self.right_size += len(word)
        self.right_items.append(word)
        
    def empty_left(self):
        self.left_size = 0
        self.left_items = []
    
    def empty_right(self):
        self.right_size = 0
        self.right_items = []
        
    def move_from_right_to_left(self):
        for item in self.right_items:
            self.add_left(item)
        self.empty_right()
        
    def move_from_left_to_right(self):
        for item in self.left_items:
            self.add_right(item)
        self.empty_left()

In [72]:
class ReflowBuffer:
    def __init__(self, max_size):
        self.max_size = max_size
        self.words = DoubleBuffer()

    def will_have_equal_dashes(self, word):
        return self.ratio_dashes(self.words.num_letters() + len(word), len(self.words.left_items) + 1) % 1 == 0

    def size_with_spaces(self):
        return self.words.num_letters() + (self.words.num_items() - 1)
    
    def can_fit(self, word):
        return self.size_with_spaces() + len(word) + 1 <= self.max_size
    
    def add_word(self, word):
        if self.will_have_equal_dashes(word):
            self.words.add_left(word)
        
        else:
            self.words.add_right(word)

    def ratio_dashes(self, num_letters, num_words):
        if num_words < 2:
            return 0
    
        return (self.max_size - num_letters) / (num_words - 1)
        
    def make_reflow_line(self):
        num_dashes = int(self.ratio_dashes(self.words.left_size, len(self.words.left_items)))
        result = (num_dashes * '-').join(self.words.left_items)
        self.words.empty_left()
        self.words.move_from_right_to_left()
        return result
    
    def make_final_reflow(self):
        self.words.move_from_right_to_left()
        return self.make_reflow_line()

I'm leaving it off right now with make final reflow in an odd state. What it needs to be is a recursive function that will pull items off of right and into left one at a time as if we were iterating it normally, then "

In [73]:
def reflow_string(s, max_size):
    buffer = ReflowBuffer(max_size)
    for word in s.split():
        if not buffer.can_fit(word):
            print(buffer.make_reflow_line())
        
        buffer.add_word(word)

    print(buffer.make_final_reflow())

In [74]:
reflow_string("asd a a a", 8)

asd----a
a------a


In [38]:
r = ReflowBuffer(8)

In [39]:
r.add_word("asd")

3 1


In [40]:
r.add_word('a')

4 2


In [41]:
r.make_reflow_line()

4 2


'asd----a'

adding second a breaks the "equal spaces" rule. so we need to set a flag to false or something
Then every new word we add, we need to keep track of separately, if we can get back to the equal spaces rule while still being can_fit, then we add all the words to the main buffer.
otherwise if we fail can_fit without passing equal spaces, we throw out all the separate words without ever adding them.

How to keep track of separate words?
I was thinking another buffer!
all we need is a place to hold them, and to check can_fit and spaces rule as if they were a part of the main thing.